In [2]:
!pip install matplotlib

  Using cached contourpy-1.3.2-cp310-cp310-macosx_10_9_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 13.2 MB/s  0:00:00 eta 0:00:01
Using cached contourpy-1.3.2-cp310-cp310-macosx_10_9_x86_64.whl (268 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 7.6 MB/s  0:00:00 eta 0:00:01m
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [matplotlib]7 [matplotlib]


In [ ]:
!python -m pip install -e ..

In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch_measure.datasets import load
from torch_measure.models import Rasch, TwoPL, ThreePL
from torch_measure.data import ResponseMatrix
from torch_measure.viz import plot_icc, plot_response_heatmap, plot_information
import torch_measure
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'torch_measure'

### Load the solver response matrix

Rows are solver models, columns are MMLU-Pro items, and values are binary correctness. Missing or unparsed responses stay as `NaN` and are masked by the fitting code.

In [ ]:
matrix_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_response_matrix.csv"
subject_meta_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_subject_metadata.csv"
item_meta_path = "../benchmarks/mmlu/response_matrices/mmlu_pro_solver_item_metadata.csv"

df = pd.read_csv(matrix_path, index_col=0)
subject_meta = pd.read_csv(subject_meta_path)
item_meta = pd.read_csv(item_meta_path)

rm = ResponseMatrix(
    data=torch.tensor(df.values, dtype=torch.float32),
    subject_ids=list(df.index.astype(str)),
    item_ids=list(df.columns.astype(str)),
    item_contents=list(item_meta["source"].astype(str)),
    subject_metadata=subject_meta.to_dict("records"),
    info={
        "benchmark_id": "mmlu_pro_solver",
        "item_id_field": "original_id",
        "value": "correct: 1.0=true, 0.0=false, NaN=unparsed",
    },
)

print(f"{rm.n_subjects} models x {rm.n_items} items, density = {rm.density:.1%}")

### Regular IRT fits

These fits keep the usual orientation: models are subjects and MMLU-Pro questions are items. The subject `ability` parameters are therefore the direct model capability estimates.

In [ ]:
regular_fits = {}

regular_specs = {
    "1pl_regular": (Rasch, {"max_epochs": 10000, "lr": 0.01}),
    "2pl_regular": (TwoPL, {"max_epochs": 10000, "lr": 0.005}),
    "3pl_regular": (ThreePL, {"max_epochs": 10000, "lr": 0.003}),
}

for fit_name, (model_cls, fit_kwargs) in regular_specs.items():
    print(f"\nFitting {fit_name}")
    model = model_cls(rm.n_subjects, rm.n_items, device=device)
    history = model.fit(
        rm.data,
        method="mle",
        verbose=True,
        **fit_kwargs,
    )
    regular_fits[fit_name] = {"model": model, "history": history}
    print(f"{fit_name} final loss: {history['losses'][-1]:.4f}")

### 1PL item-marginal MMLE

The library's built-in EM marginalizes over abilities. For model capability, this custom 1PL fit instead integrates out item difficulties and estimates only model abilities.

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm

def fit_rasch_abilities_marginalize_difficulty(
    data,
    n_quadrature=31,
    max_epochs=1000,
    lr=0.03,
    difficulty_prior_sd=1.0,
    ability_prior_weight=0.01,
    device="cpu",
):
    Y = data.to(device).float()
    mask = ~torch.isnan(Y) & (Y != -1)

    n_models, n_items = Y.shape
    ability = torch.nn.Parameter(torch.zeros(n_models, device=device))

    b_nodes, weights = np.polynomial.hermite_e.hermegauss(n_quadrature)
    b_nodes = torch.tensor(b_nodes, dtype=torch.float32, device=device) * difficulty_prior_sd
    weights = torch.tensor(weights, dtype=torch.float32, device=device)
    weights = weights / weights.sum()
    log_weights = torch.log(weights)

    opt = torch.optim.Adam([ability], lr=lr)
    losses = []

    for _ in tqdm(range(max_epochs), desc="1PL item-marginal MMLE"):
        opt.zero_grad()
        item_terms = []

        for item_idx in range(n_items):
            obs = mask[:, item_idx]
            if not obs.any():
                continue

            y = Y[obs, item_idx]
            theta = ability[obs]
            logits = theta[:, None] - b_nodes[None, :]

            logp = (
                y[:, None] * F.logsigmoid(logits)
                + (1 - y[:, None]) * F.logsigmoid(-logits)
            )
            item_terms.append(torch.logsumexp(log_weights + logp.sum(dim=0), dim=0))

        log_likelihood = torch.stack(item_terms).sum()
        prior = ability_prior_weight * ability.pow(2).mean()
        loss = -log_likelihood / len(item_terms) + prior

        loss.backward()
        opt.step()
        losses.append(loss.item())

    return ability.detach(), losses

theta_1pl_item_marginal, loss_1pl_item_marginal = fit_rasch_abilities_marginalize_difficulty(
    rm.data,
    n_quadrature=31,
    max_epochs=1000,
    lr=0.03,
    device=device,
)

print(f"1PL item-marginal final loss: {loss_1pl_item_marginal[-1]:.4f}")

### Final capability table

Accuracy is included as a sanity-check reference. The final table now includes both regular `1pl_regular` and item-marginalized `1pl_item_marginal_mmle` capability estimates.

In [ ]:
accuracy = df.mean(axis=1, skipna=True)
n_observed = df.notna().sum(axis=1)

capability_df = pd.DataFrame({
    "model": rm.subject_ids,
    "accuracy": accuracy.loc[rm.subject_ids].values,
    "n_observed": n_observed.loc[rm.subject_ids].values,
    "1pl_regular": regular_fits["1pl_regular"]["model"].ability.detach().cpu().numpy(),
    "1pl_item_marginal_mmle": theta_1pl_item_marginal.cpu().numpy(),
    "2pl_regular": regular_fits["2pl_regular"]["model"].ability.detach().cpu().numpy(),
    "3pl_regular": regular_fits["3pl_regular"]["model"].ability.detach().cpu().numpy(),
})

ability_cols = [
    "1pl_regular",
    "1pl_item_marginal_mmle",
    "2pl_regular",
    "3pl_regular",
]

for col in ability_cols:
    capability_df[f"{col}_rank"] = capability_df[col].rank(
        ascending=False,
        method="min",
    ).astype(int)

capability_df = capability_df.sort_values("1pl_regular", ascending=False)
capability_df